In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# LSTM and GRU DIAGRAM

![LSTM Model](https://daxg39y63pxwu.cloudfront.net/images/blog/lstm-model/Long_Short_Term_Memory_(LSTM)_Models.webp)

![GRU Architecture](https://media.geeksforgeeks.org/wp-content/uploads/20260305160758210335/GRU_1.webp)

# Sentence Completion

In [2]:
df = pd.read_csv("qoute_dataset.csv")

In [3]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [4]:
df.shape

(3038, 2)

In [5]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [6]:
quotes = quotes.str.lower()

In [7]:
import string
# Removing Punctuations
translator = str.maketrans('','',string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [8]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [10]:
vocab_size = 8978

tokenizer = Tokenizer(num_words = vocab_size)
tokenizer.fit_on_texts(quotes)

In [11]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10 ]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [12]:
sequence = tokenizer.texts_to_sequences(quotes)

In [13]:
quotes[0]

'“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”'

In [14]:
print(sequence[0])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]


In [15]:
X = []
y = []

for seq in sequence:
    for i in range(1, len(seq)):
      input_seq = seq[:i]
      output_seq = seq[i]
      X.append(input_seq)
      y.append(output_seq)

In [16]:
len(X)

85270

In [17]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [18]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [19]:
X_padded = pad_sequences(X,maxlen=max_len,padding = 'pre')

In [20]:
X_padded

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]], dtype=int32)

In [21]:
y = np.array(y)

In [22]:
X_padded.shape

(85270, 745)

In [23]:
y.shape

(85270,)

In [24]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y,num_classes = vocab_size)

In [25]:
y_one_hot.shape

(85270, 8978)

### Creating the model

In [26]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense,LSTM

In [27]:
embedding_dim = 50
rnn_units = 128

In [28]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_len)
)

rnn_model.add(SimpleRNN(units = rnn_units))

# O/p Layer
rnn_model.add(Dense(vocab_size,activation='softmax'))


In [29]:
rnn_model.compile(
optimizer='adam',
loss='categorical_crossentropy',
metrics=['accuracy']
)

In [30]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [31]:
lstm_model = Sequential()
lstm_model.add(Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_len))
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(vocab_size,activation='softmax'))

In [32]:
lstm_model.compile(
optimizer='adam',
loss='categorical_crossentropy',
metrics=['accuracy']
)

In [33]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [34]:
epochs = 10
batch_size = 128


In [35]:
# history = rnn_model.fit(
#     X_padded,
#     y_one_hot,
#     epochs=epochs,
#     batch_size=batch_size,
#     validation_split=0.2
# )

In [36]:
history = lstm_model.fit(
    X_padded,
    y_one_hot,
    epochs=100,
    batch_size=batch_size,
    validation_split=0.2
)

Epoch 1/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 36s 56ms/step - accuracy: 0.0385 - loss: 6.7546 - val_accuracy: 0.0429 - val_loss: 6.7356
Epoch 2/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 29s 55ms/step - accuracy: 0.0548 - loss: 6.3338 - val_accuracy: 0.0612 - val_loss: 6.6399
Epoch 3/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 42s 56ms/step - accuracy: 0.0733 - loss: 6.0910 - val_accuracy: 0.0788 - val_loss: 6.5382
Epoch 4/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 30s 57ms/step - accuracy: 0.0928 - loss: 5.8887 - val_accuracy: 0.0908 - val_loss: 6.5162
Epoch 5/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 31s 57ms/step - accuracy: 0.1047 - loss: 5.7161 - val_accuracy: 0.0942 - val_loss: 6.5122
Epoch 6/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 30s 56ms/step - accuracy: 0.1126 - loss: 5.5554 - val_accuracy: 0.0984 - val_loss: 6.5292
Epoch 7/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 31s 57ms/step - accuracy: 0.1226 - loss: 5.4006 - val_accuracy: 0.1042 - val_loss: 6.5378
Epoch 8/100
533/533 ━━━━━━━━━━━━━━━━━━━━ 31s 58ms/step - accuracy: 0.1308 - loss: 5

In [37]:
lstm_model.save("lstm_model.h5")

In [ ]:
# from tensorflow.keras.models import load_model

#lstm_model = load_model("next_word_model.h5")

In [38]:
index_to_word = {}
for word, index in word_index.items():
    index_to_word[index] = word

In [41]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [42]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])
  seq = pad_sequences(seq,maxlen=max_len,padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_word_index = np.argmax(pred)
  predicted_word = index_to_word[pred_word_index]

  return predicted_word

In [44]:
seed_text = "you will"
next_word = predictor(lstm_model,tokenizer,seed_text,max_len)
print(next_word)

never


In [47]:
def generateText(model, tokenizer, seed_text, n_words, max_len):
    for _ in range(n_words):

        next_word = predictor(
            model,
            tokenizer,
            seed_text,
            max_len
        )

        if next_word is None:
            break

        seed_text += " " + next_word

    return seed_text

In [48]:
seed_text = "The Meaning of Life"
generate_text = generateText(lstm_model,tokenizer,seed_text,10,max_len)
print(generate_text)

The Meaning of Life is to be loved is to be honorable to be


In [50]:
import pickle
with open("tokenizer.pickle","wb") as handle:
  pickle.dump(tokenizer,handle)

In [51]:
with open("max_len.pickle","wb") as handle:
  pickle.dump(max_len,handle)